In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
from scipy.stats import ttest_ind,ttest_rel,chi2_contingency,f_oneway

In [3]:
data=pd.read_csv("/content/CYBER.csv")

In [4]:
data.columns

Index(['Country', 'Year', 'Attack Type', 'Target Industry',
       'Financial Loss (in Million $)', 'Number of Affected Users',
       'Attack Source', 'Security Vulnerability Type',
       'Defense Mechanism Used', 'Incident Resolution Time (in Hours)'],
      dtype='object')

In [5]:
data.isnull().sum()

,0
Country,0
Year,0
Attack Type,0
Target Industry,0
Financial Loss (in Million $),0
Number of Affected Users,0
Attack Source,0
Security Vulnerability Type,0
Defense Mechanism Used,0
Incident Resolution Time (in Hours),0


In [8]:
data.drop_duplicates(inplace=True)

In [11]:
data["Attack Type"].unique()

array(['Phishing', 'Ransomware', 'Man-in-the-Middle', 'DDoS',
       'SQL Injection', 'Malware'], dtype=object)

***Do Ransomware and Phishing attacks cause different financial losses?***

**Independent T-Test**

In [4]:
ransomeware=data[data["Attack Type"]=="Ransomware"]["Financial Loss (in Million $)"]
phishing=data[data["Attack Type"]=="Phishing"]["Financial Loss (in Million $)"]

In [5]:
t_stat,p=ttest_ind(ransomeware,phishing)

print("T-statistics=",t_stat)
print("P-value=",p)

T-statistics= -0.4477560501925057
P-value= 0.6544243433007667


***Do attack types have different average financial losses?***

In [11]:
from scipy.stats import f_oneway

ransomware = data[
    data["Attack Type"]=="Ransomware"
]["Financial Loss (in Million $)"]

phishing = data[
    data["Attack Type"]=="Phishing"
]["Financial Loss (in Million $)"]

mitm = data[
    data["Attack Type"]=="Man-in-the-Middle"
]["Financial Loss (in Million $)"]

ddos = data[
    data["Attack Type"]=="DDoS"
]["Financial Loss (in Million $)"]

sql = data[
    data["Attack Type"]=="SQL Injection"
]["Financial Loss (in Million $)"]

malware = data[
    data["Attack Type"]=="Malware"
]["Financial Loss (in Million $)"]

f_test, p = f_oneway(
    ransomware,
    phishing,
    mitm,
    ddos,
    sql,
    malware
)

print("F-statistic =", f_test)
print("P-value =", p)

F-statistic = 0.625217695649292
P-value = 0.6805640335907177


***Do different cyber attack types result in significantly different financial losses?***

Hypotheses

H₀: Average financial loss is the same across all attack types.

H₁: At least one attack type has a different average financial loss.

The ANOVA test produced a p-value of 0.6806, which is greater than the significance level of 0.05.

Therefore, we fail to reject the null hypothesis.

There is insufficient statistical evidence to conclude that the average financial loss differs significantly across cyber attack types such as Ransomware, Phishing, DDoS, SQL Injection, Malware, and Man-in-the-Middle attacks.

Based on this dataset, attack type alone does not appear to be a major factor influencing financial loss.

Although different attack types may seem more dangerous, the financial losses associated with them are not significantly different in this dataset.

Organizations should consider other factors such as target industry, affected users, attack source, and resolution time when assessing cyber risk and financial impact.

**Is Attack Type related to Target Industry?**

In [15]:
from scipy.stats import chi2_contingency

table = pd.crosstab(
    data["Attack Type"],
    data["Target Industry"]
)

chi2,p,dof,expected = \
chi2_contingency(table)

print("Chi-square statistic =", chi2)
print("P-value =", p)

Chi-square statistic = 38.45526915816884
P-value = 0.13840727631363403


**Is Financial Loss related to Resolution Time?**

In [19]:
from scipy.stats import pearsonr

correlation, p_value = pearsonr(
    data["Financial Loss (in Million $)"],
    data["Incident Resolution Time (in Hours)"]
)

print("Correlation coefficient =", correlation)
print("P-value =", p_value)

Correlation coefficient = -0.012670791107337303
P-value = 0.4878412412040251


Hypotheses

H₀: There is no significant relationship between financial loss and incident resolution time.

H₁: There is a significant relationship between financial loss and incident resolution time.

**Fail to Reject H₀**

The correlation analysis produced a coefficient of -0.0127 and a p-value of 0.4878.

Since the p-value is greater than 0.05, there is insufficient evidence to conclude that financial loss and incident resolution time are significantly related.

The correlation coefficient is very close to zero, suggesting that incidents with higher financial losses do not necessarily take longer to resolve.

Based on this dataset, financial impact appears to be independent of incident resolution time.

Organizations should not assume that the most expensive cyber incidents will require the longest recovery periods.

Other factors such as attack complexity, security preparedness, defense mechanisms, and vulnerability type may have a stronger influence on resolution time than financial loss alone.

***Is average financial loss different from $50M?***

In [22]:
from scipy.stats import ttest_1samp

t_stat,p = ttest_1samp(
    data[
        "Financial Loss (in Million $)"
    ],
    50
)

print("T-statistics=",t_stat)
print("P-value=",p)

T-statistics= 0.9378170093543748
P-value= 0.34841397596483037


**The One-Sample T-Test produced a T-statistic of 0.938 and a p-value of 0.348.**

Since the p-value is greater than the significance level (0.05), we fail to reject the null hypothesis.

This indicates that the average financial loss from cyber attacks is not significantly different from the benchmark value of $50 million.

From a business perspective, the current benchmark remains a reasonable estimate of expected financial impact and can continue to be used for risk assessment and financial planning.

| Business Question                                | Test              | Result                        |
| ------------------------------------------------ | ----------------- | ----------------------------- |
| Do attack types have different financial losses? | ANOVA             | ❌ No Significant Difference   |
| Does financial loss affect resolution time?      | Correlation       | ❌ No Significant Relationship |
| Is average financial loss different from $50M?   | One-Sample T-Test | ❌ No Significant Difference   |
